# Camada Bronze

Ingestão dos 8 CSVs da V-Credit para tabelas Delta no schema **rocket_lab.vcredit_bronze**. Os dados são mantidos como vieram da fonte (tudo como **string**); limpeza e tipagem ficam para a camada Silver.

In [0]:
# Caminhos dos CSVs de origem, centralizados em config/paths.py
from config.paths import (
    CSV_BASE_ATENDENTES, 
    CSV_CLIENTES, 
    CSV_CUSTOS, 
    CSV_CHAMADOS,
    CSV_CANAIS, 
    CSV_BASE_MOTIVOS, 
    CSV_PESQUISA_SATISFACAO, 
    CSV_CHAMADOS_HORA
)
# Destino das tabelas: <catalog>.<schema>.<tabela>
catalog = "rocket_lab"
bronze_schema_name = "vcredit_bronze"
bronze_schema = f"{catalog}.{bronze_schema_name}"

### Ingestão dos CSVs

Atenção a três coisas:

* O **inferSchema** está desligado, então todas as colunas entram como **string**. Datas, números e IDs são convertidos na camada Silver, evitando que o Spark interprete algo errado logo na entrada;

* As bases vieram de sistemas diferentes, por isso o encoding é definido por arquivo: **clientes** e **chamados_hora** estão em **ISO-8859-1** e as demais em **utf-8**. Com o encoding errado os acentos quebram;

* Foram adicionadas as colunas **ingest_timestamp** (momento da ingestão) e **source_file** (arquivo de origem) para manter a rastreabilidade. A escrita usa **overwrite**, então reexecutar o notebook não duplica registros (a única coluna que muda é o **ingest_timestamp**).

In [0]:
from pyspark.sql.functions import current_timestamp, lit
#Garante que o CATALOGO exista
sparl.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
# Garante que o schema da Bronze exista antes de gravar as tabelas
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")



# Configuração de cada fonte:
#   caminho  -> local do CSV (vem de config.paths)
#   tabela   -> nome da tabela Bronze de destino
#   encoding -> codificação do arquivo (as bases vieram de sistemas diferentes)
#   ficheiro -> nome do arquivo, gravado na coluna de rastreabilidade source_file
config_tabelas = [
    {"caminho": CSV_BASE_ATENDENTES, "tabela": "atendentes", "encoding": "utf-8", "ficheiro": "base_atendentes.csv"},
    {"caminho": CSV_CLIENTES, "tabela": "clientes", "encoding": "ISO-8859-1", "ficheiro": "clientes.csv"},
    {"caminho": CSV_CUSTOS, "tabela": "custos", "encoding": "utf-8", "ficheiro": "custos.csv"},
    {"caminho": CSV_CHAMADOS, "tabela": "chamados", "encoding": "utf-8", "ficheiro": "chamados.csv"},
    {"caminho": CSV_CANAIS, "tabela": "canais", "encoding": "utf-8", "ficheiro": "canais.csv"},
    {"caminho": CSV_BASE_MOTIVOS, "tabela": "motivos", "encoding": "utf-8", "ficheiro": "base_motivos.csv"},
    {"caminho": CSV_PESQUISA_SATISFACAO, "tabela": "pesquisa", "encoding": "utf-8", "ficheiro": "pesquisa_satisfacao.csv"},
    {"caminho": CSV_CHAMADOS_HORA, "tabela": "chamados_hora", "encoding": "ISO-8859-1", "ficheiro": "Chamados_Hora.CSV"}
]

for config in config_tabelas:
    caminho_dados = config["caminho"]
    nome_tabela = f"{bronze_schema}.{config['tabela']}"
    ficheiro = config["ficheiro"]

    # Leitura do CSV
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("encoding", config["encoding"])
        .option("quote", '"')
        .option("escape", '"')
        .csv(caminho_dados)
    )

    # Rastreabilidade
    # ingest_timestamp: momento da ingestão | source_file: arquivo de origem do registro
    df_bronze = (
        df.withColumn("ingest_timestamp", current_timestamp())
        .withColumn("source_file", lit(ficheiro))
    )
    # Escrita idempotente 
    # overwrite: reexecutar não duplica registros
    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(nome_tabela)
    )
